#### Training GPT from Scratch on Discharge Summaries



In this lab, I will walk through a slightly modified version of this huggingface tutorial on training GPT from scratch: https://huggingface.co/learn/llm-course/en/chapter7/6

Additionally, I'll explore how training on a small corpus of discharge summaries affects the embeddings and generations of our model. I'll also primarily use the higher level "transformers" API.

(For lower-level API, check out Andrej Karpathy's implementation of minGPT: https://github.com/karpathy/minGPT/tree/master)

This notebook can be run completely locally, but it is recommend to use a GPU. If running in colab, connect to a T4 runtime.

LLM Statement: I used AI to correct my claims, and break down the decision choices for the GPT-2 training and architecture settings.

In [1]:
# !pip install transformers torch accelerate datasets

In [2]:
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import torch.nn.functional as F
import torch
import math

In [3]:
from google.colab import files

First, we'll load the data that we used in Lab 2. If using this Colab, upload the "lab2-data.csv" file. If working locally, point to the relevant directory.

In [ ]:
uploaded = files.upload()

In [ ]:
discharge_summaries = pd.read_csv('data.csv')
dataset = Dataset.from_pandas(discharge_summaries)
print(dataset)

Dataset({
    features: ['HADM_ID', 'TEXT'],
    num_rows: 3668
})


Previously, I have created my own vocabulary from the corpus. Here, I'll use a pre-trained tokenizer which unlike before, I can now compact any text, have a fixed vocabulary and eliminate unknown tokens. I'll also restrict the context length to 512, hyperparameter I can tune to control how much text the model can see at once. Finally, the tokenizer class, ```AutoTokenizer``` allows us to break our large discharge summaries into smaller chunks that fit into our context limit.

In [6]:
context_length = 512
tokenizer = AutoTokenizer.from_pretrained("gpt2")

outputs = tokenizer(
    dataset[:10]["TEXT"],
    truncation=True,
    max_length=context_length,
    return_overflowing_tokens=True,
    return_length=True,
)

print(f"Input IDs length: {len(outputs['input_ids'])}")
print(f"Input chunk lengths: {(outputs['length'])}")
print(f"Chunk mapping: {outputs['overflow_to_sample_mapping']}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Input IDs length: 73
Input chunk lengths: [512, 512, 512, 512, 512, 512, 512, 512, 278, 512, 512, 512, 512, 512, 512, 512, 512, 512, 207, 512, 512, 512, 512, 512, 512, 512, 512, 315, 512, 110, 512, 512, 512, 512, 295, 512, 512, 512, 512, 512, 512, 512, 512, 475, 512, 512, 512, 512, 512, 115, 512, 512, 512, 512, 512, 512, 110, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 196, 512, 512, 512, 512, 437]
Chunk mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9]


Each chunk will be list of indices corresponding to tokens in our vocabulary.

In [7]:
print(outputs['input_ids'][0])

[2782, 3411, 7536, 25, 220, 685, 1174, 17, 11623, 12, 17, 12, 24, 1174, 60, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 3167, 10136, 7536, 25, 220, 220, 685, 1174, 17, 11623, 12, 17, 12, 1433, 1174, 60, 628, 198, 16177, 25, 26112, 2149, 8881, 198, 198, 3237, 6422, 444, 25, 198, 57, 420, 273, 1220, 406, 3798, 349, 198, 198, 8086, 1571, 33250, 1174, 37564, 4586, 6530, 1248, 3553, 1174, 60, 198, 23675, 20011, 2913, 25, 198, 45170, 2356, 198, 198, 24206, 311, 31839, 393, 10001, 17443, 34997, 25, 198, 30645, 8710, 516, 1627, 36075, 357, 3506, 5387, 45808, 934, 27208, 8, 198, 198, 18122, 286, 21662, 5821, 1108, 25, 198, 5246, 13, 685, 1174, 29870, 938, 3672, 1248, 3365, 1174, 60, 318, 281, 9508, 27406, 582, 351, 10768, 257, 419, 291, 45219, 5958, 357, 43435, 198, 49257, 9809, 287, 685, 1174, 17, 17464, 1174, 60, 351, 685, 1174, 14749, 357, 403, 8, 16003, 1174, 60, 352, 12067, 17, 11, 31312, 2579, 8085, 39, 70, 11, 10768, 198, 2781, 1373, 842, 3686, 3780, 11, 11607, 257, 

To see some of what those indices mean, we can convert the ids to tokens:

In [8]:
tokenizer.convert_ids_to_tokens([2782, 3411])

['Ad', 'mission']

We can also decode the whole list back to text as well:

In [9]:
tokenizer.decode(outputs['input_ids'][0])

'Admission Date:  [**2125-2-9**]              Discharge Date:   [**2125-2-16**]\n\n\nService: MEDICINE\n\nAllergies:\nZocor / Lescol\n\nAttending:[**Doctor Last Name 1857**]\nChief Complaint:\nChest pain\n\nMajor Surgical or Invasive Procedure:\nCentral venous line insertion (right internal jugular vein)\n\nHistory of Present Illness:\nMr. [**Known lastname 1858**] is an 84 yo man with moderate aortic stenosis (outside\nhospital echo in [**2124**] with [**Location (un) 109**] 1 cm2, gradient 28 mmHg, moderate\nmitral regurgitation, mild aortic insufficiency), chronic left\nventricular systolic heart failure with EF 25-30%, hypertension,\nhyperlipidemia, diabetes mellitus, CAD s/p CABG in [**2099**] with\nSVG-LAD-Diagonal, SVG-OM, and SVG-RPDA-RPL, with a re-do CABG in\n[**9-/2117**] with LIMA-LAD, SVG-OM, SVG-diagonal, and SVG-RCA. He also\nhas severe peripheral arterial disease s/p peripheral bypass\nsurgery. He presented to [**Hospital 1474**] Hospital ER this morning with\nshortness

Let's now apply the tokenizer across the whole dataset. Note the setting for this can be changed depending on how you want to customize your tokenizer.

In [10]:
def tokenize(element):
    outputs = tokenizer(
        element["TEXT"],
        truncation=True,
        max_length=context_length, # hyperparameter, 512 in our case
        # you don't want to keep the first 512 tokens of
        # each note and throw the rest out, so set the below as True!
        # Nice to have: can experiment with stride=... so that you can have
        # the chunks overlap
        return_overflowing_tokens=True,
        return_length=True,
    )
    return {"input_ids": outputs['input_ids']}


tokenized_dataset = dataset.map(
    tokenize, batched=True, remove_columns=dataset.column_names
)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

The goal of this case study is train a language model froom *scratch*, so I'll load in a randomly initialized model with the GPT2 architecture.

In [11]:
config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=context_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    # More layers give the model more capacity to learn complex patterns,
    # but make it slower and more memory-hungry. With a small dataset,
    # a deep model is also more likely to overfit.
    n_layer=6,
    n_head=6 # multi-head attention aspect
    # note we will be using GPT-2's default n_embd=768

)
model = GPT2LMHeadModel(config)

Let's explore this architecture a bit further:

In [12]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

What do WTE and WPE refer to?

As we can see from the model, WTE and WPE has embedding (50257, 768) and (1024, 768), respectively. Since a transformer cannot read integers directly, Word Token Embeddings (WTE) and Word Position Embeddings (WPE) are used to turn discrete IDs into dense vectors that can be used by GPT-2. One way to think of this would be that WTE encodes what the token is, while WPE encodes where it is. More specifically, WTE is a lookup table with one learned 768-dimension vector for each of the GPT-2 BPE tokens, which comes out to be 50,257 BPE tokens. Then, WPE is another look table with one learned 768-dimension vector for each of the position in the sequence. The 1024 comes from the default number of positions for GPT-2.

Similar to case study 2, I can search for similar tokens using vector distances in the embedding space.

In [13]:
tokens = tokenizer.encode('hospital')
result_vector = model.transformer.wte.weight[tokens].mean(axis=0)

similarities = F.cosine_similarity(
    result_vector,
    model.transformer.wte.weight
)
top_indices = similarities.topk(10).indices
print([tokenizer.decode(idx) for idx in top_indices if idx not in tokens])

[' ideological', 'BUS', ' admit', ' Passion', '148', ' preferably', ' relegated', 'entials', ' wanted']


Why are we getting these tokens?

I encoded "hospital" and then I wanted to know what are the similar tokens are. The top 9 most similar tokens are shown in the list. The nearest tokens appear to not have aclear relationship to "hospital". The reason is actually expected because WTE has been built with random weights. My guess that this should improve after the training.

See what it can generate right now, before training.

In [14]:
token_ids = model.generate(max_length=100)
token_ids

tensor([[50256, 49738, 41346, 41346, 41346, 41346, 41346, 41346, 23818,  7158,
         50039, 50039, 50039, 50039, 50039, 50039, 50039, 50039, 50039, 50039,
         14056, 14056, 50039, 50039, 50039, 21067, 37540, 37540, 37540, 37540,
         37540, 37309, 37309, 37309, 37309,  9218, 20399, 17220, 17220, 16484,
         10230, 36404, 49707, 49707, 32457, 32457, 39025, 10392, 10392, 10392,
         10392,  7689,  7689,  7689,  7689,  7456, 22793, 22793, 22793, 22793,
         22793, 22793, 22793, 48242, 29151, 15590, 15590, 33622, 12041, 18748,
         13996, 13996, 41214, 50164, 23880,  8741,  8741,  8741,  8741,  8741,
          8741,  8741,  8741, 26728, 26728, 23550, 37408, 26728, 26728, 26728,
         22793, 11636, 15590, 15590, 15590, 15590, 15590, 15590, 15590, 15590]])

In [15]:
len(token_ids[0])

100

In [16]:
tokenizer.decode(token_ids[0])

'<|endoftext|>Spect Tayyip Tayyip Tayyip Tayyip Tayyip Tayyip cumulative Kennactualactualactualactualactualactualactualactualactualactual Truth Truthactualactualactualheeresourcesresourcesresourcesresourcesresources414414414414Key processors Remove Removeanga buck Ion HOL HOLXLXL amassed packages packages packages packagesblesblesblesblesuhirrelirrelirrelirrelirrelirrelirrel porous Jewel Assault AssaultMovingPaulilitary cure cureippystories Kurds Nation Nation Nation Nation Nation Nation Nation Nation swings swings mins lush swings swings swingsirrel speakers Assault Assault Assault Assault Assault Assault Assault Assault'

So far without training, the output seems to be giving random tokens. Again, this is as expected because the model's weights are still randomly initialized. However, one might see that there's a lot of repetition, and likely because the default decoding strategy always picks the highest-scoring token. Ultimately, the output here shows that using the architecture alone will not grant the model's language ability. This comes from training.

##### Forward Pass

Let's pass one of the input vectors in and explore the outputs.

First, I'll past the tokenized input and retrieve the embeddings.

In [17]:
tokenized_input = torch.tensor(outputs['input_ids'][0:1])
token_embeddings = model.transformer.wte(tokenized_input)
print(token_embeddings)

tensor([[[ 3.3956e-02,  7.1986e-05, -2.0779e-03,  ..., -9.0356e-03,
          -9.1455e-03, -7.3621e-04],
         [-6.5005e-03,  2.8836e-02,  1.2462e-02,  ..., -6.0816e-03,
           2.0687e-02, -4.0690e-03],
         [-3.3448e-03,  3.8637e-02,  3.9613e-02,  ..., -6.1030e-02,
          -2.5832e-03,  1.1092e-02],
         ...,
         [ 2.4544e-02, -2.6876e-02,  2.3104e-03,  ..., -5.0568e-03,
          -1.1737e-02,  2.4087e-02],
         [-1.3123e-02, -2.4254e-02,  2.9640e-02,  ..., -5.4028e-03,
          -3.2964e-02, -2.7609e-02],
         [ 3.0305e-02, -3.8084e-04, -1.0262e-02,  ..., -2.2024e-02,
          -6.9954e-04, -7.5621e-03]]], grad_fn=<EmbeddingBackward0>)


What is th size of the WTE?

In [18]:
token_embeddings.shape

torch.Size([1, 512, 768])

The size of the embedding space is 768.

What is the size of the WPE?

In [19]:
# wpe is index by position numbers
# since we're using context size of 512
# the wpe will have positions from 0 to 511
cxt_len = tokenized_input.size(1)
cxt_len

512

In [20]:
# added device for running on colab
positions = torch.arange(cxt_len, device=tokenized_input.device).unsqueeze(0)
positions

tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
          28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
          42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
          56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
          70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
          84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
          98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
         112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
         126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
         140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
         154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
         168, 169, 170, 171, 172, 173, 174, 175, 176

In [21]:
position_embeddings = model.transformer.wpe(positions)
position_embeddings

tensor([[[ 0.0116, -0.0002, -0.0052,  ...,  0.0251, -0.0343,  0.0190],
         [-0.0001,  0.0393,  0.0145,  ..., -0.0051, -0.0040, -0.0058],
         [ 0.0057,  0.0086,  0.0051,  ...,  0.0285, -0.0102,  0.0235],
         ...,
         [ 0.0012, -0.0081,  0.0398,  ..., -0.0331,  0.0162, -0.0216],
         [-0.0070, -0.0238, -0.0024,  ...,  0.0359,  0.0402, -0.0187],
         [-0.0108,  0.0221, -0.0071,  ..., -0.0178,  0.0162,  0.0102]]],
       grad_fn=<EmbeddingBackward0>)

In [22]:
position_embeddings.shape

torch.Size([1, 512, 768])

As expected, our position embedding captures the number of positions to use, 512, which was defined earlier as ```context_length``` from our configuration and 768 is default embedding size for GPT-2.

How do we combine using WTE and WPE? 

In [23]:
hidden_states = token_embeddings + position_embeddings
hidden_states.shape # expect (1, 512, 768)

torch.Size([1, 512, 768])

Now we'll layer normalize. Our transformer has 6 attention heads, but what does it look like?

In [24]:
normalized_hidden_states = model.transformer.h[0].ln_1(hidden_states)

print("Unnormalized:")
print(hidden_states)
print('-----------------')
print("Normalized:")
print(normalized_hidden_states)

Unnormalized:
tensor([[[ 0.0455, -0.0001, -0.0073,  ...,  0.0161, -0.0435,  0.0183],
         [-0.0066,  0.0682,  0.0270,  ..., -0.0112,  0.0167, -0.0098],
         [ 0.0024,  0.0473,  0.0447,  ..., -0.0325, -0.0128,  0.0346],
         ...,
         [ 0.0258, -0.0350,  0.0421,  ..., -0.0381,  0.0044,  0.0025],
         [-0.0201, -0.0481,  0.0273,  ...,  0.0305,  0.0072, -0.0463],
         [ 0.0195,  0.0217, -0.0173,  ..., -0.0398,  0.0155,  0.0026]]],
       grad_fn=<AddBackward0>)
-----------------
Normalized:
tensor([[[ 1.5719, -0.0049, -0.2534,  ...,  0.5553, -1.5020,  0.6304],
         [-0.2639,  2.4095,  0.9373,  ..., -0.4271,  0.5700, -0.3785],
         [ 0.0873,  1.7142,  1.6230,  ..., -1.1763, -0.4615,  1.2569],
         ...,
         [ 1.0388, -1.1471,  1.6249,  ..., -1.2591,  0.2713,  0.2010],
         [-0.7556, -1.7672,  0.9550,  ...,  1.0716,  0.2318, -1.7011],
         [ 0.7482,  0.8273, -0.5753,  ..., -1.3836,  0.6043,  0.1430]]],
       grad_fn=<NativeLayerNormBackward0>

Let's take a look at the self-attention matrices

In [25]:
model.transformer.h[0].attn.c_attn.weight.shape

torch.Size([768, 2304])

One key thing to notice is the 2304 dimension. Where is it coming from? That is precisely $3 \times 768 = 2304$, where the 3 represents the three matrices: query, key, and value, which are used in self-attention.

In practice, these three matrices are often stacked into a single matrix for efficient computation. So the matrix above represents [Wq Wk Wv].

We can derive our Q, K, V matrices by splitting the output:

In [26]:
Q, K, V = model.transformer.h[0].attn.c_attn(normalized_hidden_states).split(768, dim=2)
print('Q:', Q)
print('K:', K)
print('V:', V)

Q: tensor([[[-0.1267,  0.5345,  0.2149,  ...,  0.2206, -0.1625, -0.2830],
         [ 0.1191, -1.5661,  0.1626,  ..., -0.7709, -0.3244, -0.1549],
         [-0.4097,  0.3068,  0.6022,  ..., -0.6456,  0.4809, -0.1363],
         ...,
         [ 0.8504, -0.6792, -0.5394,  ..., -0.4595, -0.6679, -0.5224],
         [ 0.1841, -0.5574,  0.7298,  ...,  0.4753, -0.5959, -0.4961],
         [ 0.5116,  0.8685,  0.4462,  ...,  0.2622, -0.4006,  0.5519]]],
       grad_fn=<SplitBackward0>)
K: tensor([[[ 0.4953, -0.3750, -0.0021,  ..., -0.9494, -0.0126,  0.2842],
         [-0.4302, -0.3482, -0.6362,  ..., -0.2517,  0.4184,  0.5027],
         [ 0.8283, -0.2785,  0.8594,  ...,  0.3083, -0.1713, -0.4844],
         ...,
         [-0.2034,  1.0668, -0.0816,  ..., -0.0921,  0.9864,  0.1692],
         [ 0.4828, -0.4874,  0.2741,  ..., -0.1917, -0.9460,  0.0612],
         [-0.1360,  0.6930, -0.5904,  ...,  0.6276, -0.1386, -0.0096]]],
       grad_fn=<SplitBackward0>)
V: tensor([[[ 0.8090, -0.2668, -0.5494,  ...

In [27]:
print(Q.shape)

torch.Size([1, 512, 768])


These matrices are random! They only gain significance during training because the following constraint is applied during the forward pass:

In [28]:
att = (Q @ K.transpose(-2, -1)) * (1.0 / math.sqrt(K.size(-1)))
print("QK^T Dim:", att.shape)
A = F.softmax(att, dim=-1)
Z = A @ V
print("Z Dim:", Z.shape)

QK^T Dim: torch.Size([1, 512, 512])
Z Dim: torch.Size([1, 512, 768])


Finally, the output of the model will be a matrix of logits for each position in the sequence. Recall that GPT-2's BPE tokeniziation size is 50,257. During training, U will compute the cross entropy between that logit and the embedding vectors of the next tokens.

In [29]:
model(tokenized_input).logits.shape

torch.Size([1, 512, 50257])

##### Model Training

The chunks are not all the same, and since a tensor can't have different lengths, that's what padding is for.

In [30]:
tokenizer.pad_token = tokenizer.eos_token
# sets pad position so that model will ignore it instead learning the padding

# converts this to tensors for the model to use for training
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [31]:
args = TrainingArguments(
    output_dir="lab3",
    # bigger batches give smoother gradients, but uses more GPU memory
    per_device_train_batch_size=14, # increase/decrease this based on your memory
    eval_steps=50,
    logging_steps=50,
    # 1 means no accumulation, and it gets updated every batch
    gradient_accumulation_steps=1,
    # passes over the whole dataset twice
    num_train_epochs=2,
    # regularization, pulls weight toward 0 to reduce overfitting
    # 0.1 is the standard value
    weight_decay=0.1,
    # ramp the learning rate up, this is to prevent starting at full speed that could destabilize training
    warmup_steps=10,
    lr_scheduler_type="cosine",
    # this step size is bigger than usual (5e-5) because we're training from scratch
    learning_rate=5e-4,
    fp16=True,
    report_to="none"
    # use_cpu=True # Very slow! Feel free to use without a GPU if you'd like
)

# let's you write the training loop through Hugging Face
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset
)

The model will internally handle the next-token prediction loss. On the T4 GPU, this took me roughly 40 minutes.

In [32]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,7.072075
100,5.508832
150,4.943105
200,4.464794
250,4.124619
300,3.873821
350,3.716230
400,3.512885
450,3.397010
500,3.309206


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3926, training_loss=2.4383374105833404, metrics={'train_runtime': 2357.4114, 'train_samples_per_second': 23.306, 'train_steps_per_second': 1.665, 'total_flos': 7178083035512832.0, 'train_loss': 2.4383374105833404, 'epoch': 2.0})

Now, what are the trained embeddings most similar to "hospital"? How do they compare to the tokens from earlier?

In [33]:
# reusing the code from Q2, earlier we didn't train yet, now we trained it
tokens = tokenizer.encode('hospital')
result_vector = model.transformer.wte.weight[tokens].mean(axis=0)
similarities = F.cosine_similarity(result_vector, model.transformer.wte.weight)
top_indices = similarities.topk(10).indices
print([tokenizer.decode(idx) for idx in top_indices if idx not in tokens])

[' hospital', 'stay', 'transfer', 'present', 'medical', 'being', 'condition', 'author', 'morning']


Now, the tokens are in an embedding space that reflects the data. it was trained on, discharge summaries. Unlike when we looked at the tokens from earlier, which was based on random initializations, most of the tokens now have related meaning to "hospital".

Let's generate a few samples and compare those results from earlier.

In [34]:
few = 3
for i in range(few):
    output_tokens = model.generate(max_length=100, do_sample=True, temperature=0.1)
    print(i, tokenizer.decode(output_tokens[0]))

[transformers] We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


0 <|endoftext|>, and
was treated with a heparin drip.  He was started on a heparin drip
and was started on a heparin drip.  He was then transferred to the CCU for
further management.

PAST MEDICAL HISTORY:
1.  Hypertension.
2.  Hypercholesterolemia.
3.  Hypercholesterolemia.
4.  Hypercholesterolemia.
5. 
1 <|endoftext|>

[**2172-1-9**]
[**2172-1-9**] 06:00AM BLOOD WBC-9.9 RBC-3.51* Hgb-8.5* Hct-28.0*
MCV-93 MCH-30.9 MCHC-33.7 RDW-14.2 Plt Ct-206
[**2172-1-9**] 06:00
2 <|endoftext|>
with a new diagnosis of aortic stenosis.  The patient was
transferred to the CCU for further management.

PAST MEDICAL HISTORY:
1.  Coronary artery disease status post coronary artery bypass graft
graft times two.
2.  Hypertension.
3.  Hypercholesterolemia.
4.  Status post right carotid artery stent.
5.  Status post right carotid


Earlier when tasked to generate examples, it gave random, repetitive tokens, now it is able to emulate what the discharge summary structure looks like, perhaps due to the fact that $T < 1$ sharpens the distribution. There are still some limitations. For example, there are still some repetition, and perhaps that has more to do with the fact that we only trained with 2 epochs.